# MMGMT 722 – Assignment 1: Finding the Best Delivery Promise for Rosa's Pizza

##Set up

In [1]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

print("ZONES:", ZONES)
print("TIME_BLOCKS:", TIME_BLOCKS)
print("COSTS:", COSTS)
print("PROMISE:", PROMISE)

ZONES: ['Central', 'North', 'Far West']
TIME_BLOCKS: ['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
COSTS: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
PROMISE: 45


In [3]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)
print(len(times), "orders; average delivery time", round(times.mean(), 1), "minutes")
print("First 10 delivery times:", times[:10])

210 orders; average delivery time 42.9 minutes
First 10 delivery times: [46.6 53.7 46.4 28.4 55.1 48.  35.8 50.  46.9 45.9]


## Part I. Describe: How bad is it, really?
### (a) Percentage of late orders
An order is late when its delivery time exceeds the promised time.

In [4]:
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=1)

is_late = times > 45          # array of True/False, one per order
print("First 10 late flags:", is_late[:10])

num_late = is_late.sum()      # True counts as 1, False as 0
num_orders = len(times)
late_pct = num_late / num_orders * 100

print("Late orders:", num_late, "out of", num_orders)
print("Late rate:", round(late_pct, 1), "%")

First 10 late flags: [ True  True  True False  True  True False  True  True  True]
Late orders: 77 out of 210
Late rate: 36.7 %


In [5]:
def get_times(zone, time_block, promise, seed=1):
    """Return all delivery times (minutes) for the given zone and time block.
    zone or time_block can be 'all' to include every zone / time block."""
    zones = ZONES if zone == 'all' else [zone]
    blocks = TIME_BLOCKS if time_block == 'all' else [time_block]

    all_times = []
    for z in zones:
        for tb in blocks:
            all_times.append(delivery_times(z, tb, promise, seed=seed))

    return np.concatenate(all_times)   # join the arrays into one big array


def late_rate(zone, time_block, promise, seed=1):
    """Percentage of orders delivered later than the promise."""
    times = get_times(zone, time_block, promise, seed)
    return (times > promise).mean() * 100

get_times gathers the delivery times for the requested zone(s) and time block(s), pooling orders when 'all' is used, so larger segments get more weight. late_rate returns the share of those orders that arrived after the promise.

In [6]:
print("Far West, Fri/Sat eve:", round(late_rate("Far West", "Fri/Sat eve", 45), 1), "%")
print("All zones, Lunch:     ", round(late_rate("all", "Lunch", 45), 1), "%")
print("North, all blocks:    ", round(late_rate("North", "all", 45), 1), "%")
print("Overall:              ", round(late_rate("all", "all", 45), 1), "%")

Far West, Fri/Sat eve: 36.7 %
All zones, Lunch:      3.3 %
North, all blocks:     6.8 %
Overall:               8.3 %


### (b) Ranking zone/time-block pairs by late rate (45-minute promise)

In [7]:
late_results = []
for z in ZONES:
    for tb in TIME_BLOCKS:
        rate = late_rate(z, tb, 45)
        n_orders = len(delivery_times(z, tb, 45, seed=1))
        late_results.append((z, tb, rate, n_orders))

# sort by the late rate (position 2 in each tuple), highest first
late_results.sort(key=lambda row: row[2], reverse=True)

print(f"{'Rank':<5}{'Zone':<10}{'Time block':<14}{'Late %':>8}{'Orders':>8}")
for rank, (z, tb, rate, n) in enumerate(late_results, start=1):
    print(f"{rank:<5}{z:<10}{tb:<14}{rate:>7.1f}%{n:>8}")

Rank Zone      Time block      Late %  Orders
1    Far West  Fri/Sat eve      36.7%     210
2    North     Fri/Sat eve      16.7%     150
3    Far West  Weekday eve      11.2%     170
4    Central   Fri/Sat eve       6.8%     190
5    North     Weekday eve       6.0%     250
6    Far West  Other             4.7%     192
7    Central   Weekday eve       4.2%     330
8    North     Lunch             3.7%     190
9    Central   Lunch             3.5%     260
10   Central   Other             2.6%     190
11   North     Other             2.5%     160
12   Far West  Lunch             2.5%     120


### (c) Average delivery time

In [8]:
def avg_time(zone, time_block, promise, seed=1):
    """Average delivery time (minutes) for the given zone and time block.
    zone or time_block can be 'all'."""
    times = get_times(zone, time_block, promise, seed)
    return times.mean()


print("Far West, Fri/Sat eve:", round(avg_time("Far West", "Fri/Sat eve", 45), 1), "min")
print("All zones, Lunch:     ", round(avg_time("all", "Lunch", 45), 1), "min")
print("North, all blocks:    ", round(avg_time("North", "all", 45), 1), "min")
print("Overall:              ", round(avg_time("all", "all", 45), 1), "min")

Far West, Fri/Sat eve: 42.9 min
All zones, Lunch:      28.2 min
North, all blocks:     31.0 min
Overall:               31.2 min


### (d) Ranking zone/time-block pairs by average delivery time (45-minute promise)

In [9]:
late_rank = {(z, tb): rank for rank, (z, tb, rate, n) in enumerate(late_results, start=1)}

avg_results = []
for z in ZONES:
    for tb in TIME_BLOCKS:
        avg_results.append((z, tb, avg_time(z, tb, 45), late_rate(z, tb, 45)))

avg_results.sort(key=lambda row: row[2], reverse=True)

print(f"{'Rank':<5}{'Zone':<10}{'Time block':<14}{'Avg min':>8}{'Late %':>8}{'Late rank':>11}")
for rank, (z, tb, avg, rate) in enumerate(avg_results, start=1):
    print(f"{rank:<5}{z:<10}{tb:<14}{avg:>8.1f}{rate:>7.1f}%{late_rank[(z, tb)]:>11}")

Rank Zone      Time block     Avg min  Late %  Late rank
1    Far West  Fri/Sat eve       42.9   36.7%          1
2    North     Fri/Sat eve       36.8   16.7%          2
3    Far West  Weekday eve       35.0   11.2%          3
4    Central   Fri/Sat eve       32.1    6.8%          4
5    North     Weekday eve       31.4    6.0%          5
6    Far West  Lunch             30.6    2.5%         12
7    Far West  Other             30.0    4.7%          6
8    Central   Weekday eve       29.0    4.2%          7
9    North     Lunch             28.7    3.7%          8
10   North     Other             27.7    2.5%         11
11   Central   Lunch             26.8    3.5%          9
12   Central   Other             26.0    2.6%         10


### Which measure, the late rate (b) or the average delivery time (d), is more relevant to Rosa's decisions? Explain why, using your results from b and d.

The late rate is more relevant to Rosa's decisions than the average delivery time. Rosa's costs come from individual late orders: each order delivered after the promise triggers a $10 refund and leads to 1.8 lost future orders. An early delivery does not cancel out a late one, but an average lets it. The average therefore hides the problem. All 12 zone/time-block pairs have averages below the 45-minute promise, including Far West on Fri/Sat evenings (42.9 minutes), so averages alone would suggest that Rosa is keeping her promise everywhere. In fact, 36.7% of Far West Fri/Sat orders, about 77 of 210, arrive late.

The two rankings also disagree in ways that matter. Far West Lunch ranks 6th by average time (30.6 minutes) but has the lowest late rate of all (2.5%). Central Lunch, by contrast, has a faster average (26.8 minutes) but a higher late rate (3.5%). This happens because the average describes only the typical delivery, while lateness depends on how many deliveries fall in the slow tail beyond 45 minutes. Since the late rate measures exactly the event that costs Rosa money, it should guide her decisions. Combined with the number of orders (late rate × orders = number of late orders), it shows where the losses are concentrated: Far West and North on Fri/Sat evenings.

## Part II. Exploring the best promise
### (a) Total cost per late order
Each late order costs Rosa the refund plus the profit lost from future orders
the customer no longer places: cost = refund + churn_orders × margin.

In [10]:
def late_cost(costs):
    """Total cost ($) of one late order: refund + profit lost to churn."""
    return costs['refund'] + costs['churn_orders'] * costs['margin']


print("Refund:              $", COSTS['refund'])
print("Lost profit (churn): $", round(COSTS['churn_orders'] * COSTS['margin'], 2))
print("Total per late order: $", round(late_cost(COSTS), 2))

Refund:              $ 10.0
Lost profit (churn): $ 16.2
Total per late order: $ 26.2


### (b) Choosing the best promise
Net profit = orders × margin − late orders × cost per late order.
A shorter promise attracts more orders but makes more of them late;
a longer promise reduces lateness but loses customers.

In [11]:
def net_profit(zone, time_block, promise, costs, seed=1):
    """Net profit ($) over four weeks for one zone, time block and promise."""
    times = delivery_times(zone, time_block, promise, seed=seed)
    n_orders = len(times)
    n_late = (times > promise).sum()
    return n_orders * costs['margin'] - n_late * late_cost(costs)


# Explore a wide range for one segment to see the shape
print(f"{'Promise':>8}{'Orders':>8}{'Late %':>9}{'Net profit':>12}")
for p in range(5, 95, 5):
    t = delivery_times("Far West", "Fri/Sat eve", p, seed=1)
    print(f"{p:>8}{len(t):>8}{(t > p).mean()*100:>8.1f}%{net_profit('Far West', 'Fri/Sat eve', p, COSTS):>12.0f}")

 Promise  Orders   Late %  Net profit
       5     242   100.0%       -4162
      10     242   100.0%       -4162
      15     241   100.0%       -4145
      20     241    99.2%       -4093
      25     239    97.5%       -3954
      30     237    92.0%       -3579
      35     232    77.2%       -2602
      40     224    59.4%       -1469
      45     210    36.7%        -127
      50     188    17.6%         827
      55     158     5.1%        1212
      60     121     1.7%        1037
      65      84     0.0%         756
      70      54     0.0%         486
      75      32     0.0%         288
      80      18     0.0%         162
      85      10     0.0%          90
      90       6     0.0%          54


**Choosing the range of promises.** The range should cover every promise that
could plausibly be best, and nothing more. At the low end, promises shorter than
Rosa can physically deliver make nearly every order late (e.g. Far West Fri/Sat:
92% late at 30 min, net profit −$3,579), so there is no point going much below
the fastest segments' typical delivery times (Central averages about 26–27 min
at lunch and off-peak). At the high end, once the late rate reaches about 0%, a
longer promise only loses customers (Far West Fri/Sat: 0% late from 65 min,
and profit falls from $756 to $54 by 90 min), so profit can only decrease.
We therefore search from 20 to 90 minutes in 5-minute steps, which also
includes today's 45-minute promise as a baseline. As a check, the best promise
should never land at either end of the range; if it does, the range should be
extended.

In [12]:
PROMISES = list(range(20, 95, 5))   # 20, 25, ..., 90
print(PROMISES)

[20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90]


In [13]:
def best_promise(zone, time_block, promises, costs, seed=1):
    """Try each promise and return (best promise, its net profit, all profits).
    `all profits` is a dict {promise: net profit}, handy for charts in the app."""
    profits = {}
    for p in promises:
        profits[p] = net_profit(zone, time_block, p, costs, seed)

    best_p = max(profits, key=profits.get)   # promise with the highest profit

    # Warn if the best promise sits at an edge of the range
    if best_p == min(promises) or best_p == max(promises):
        print(f"Warning: best promise for {zone}, {time_block} is at the edge "
              f"of the range ({best_p} min). Consider widening the range.")

    return best_p, profits[best_p], profits

In [14]:
p, profit, all_profits = best_promise("Far West", "Fri/Sat eve", PROMISES, COSTS)
current = all_profits[45]

print(f"Far West, Fri/Sat eve")
print(f"  Recommended promise: {p} minutes")
print(f"  Net profit:          ${profit:,.0f}")
print(f"  Today (45 min):      ${current:,.0f}")
print(f"  Improvement:         ${profit - current:,.0f} over four weeks")

Far West, Fri/Sat eve
  Recommended promise: 55 minutes
  Net profit:          $1,212
  Today (45 min):      $-127
  Improvement:         $1,340 over four weeks


### Recommendations for every zone and time block

In [15]:
print(f"{'Zone':<10}{'Time block':<14}{'Best':>6}{'Profit':>10}{'At 45':>10}{'Gain':>9}")

total_best, total_now = 0, 0
for z in ZONES:
    for tb in TIME_BLOCKS:
        p, profit, all_profits = best_promise(z, tb, PROMISES, COSTS)
        now = all_profits[45]
        total_best += profit
        total_now += now
        print(f"{z:<10}{tb:<14}{p:>6}{profit:>10,.0f}{now:>10,.0f}{profit - now:>9,.0f}")

print("-" * 59)
print(f"{'Total':<30}{total_best:>10,.0f}{total_now:>10,.0f}{total_best - total_now:>9,.0f}")

Zone      Time block      Best    Profit     At 45     Gain
Central   Lunch             45     2,104     2,104        0
Central   Weekday eve       45     2,603     2,603        0
Central   Fri/Sat eve       50     1,399     1,369       30
Central   Other             45     1,579     1,579        0
North     Lunch             45     1,527     1,527        0
North     Weekday eve       45     1,857     1,857        0
North     Fri/Sat eve       50       996       695      301
North     Other             45     1,335     1,335        0
Far West  Lunch             45     1,001     1,001        0
Far West  Weekday eve       50     1,211     1,032      179
Far West  Fri/Sat eve       55     1,212      -127    1,340
Far West  Other             45     1,492     1,492        0
-----------------------------------------------------------
Total                             18,317    16,468    1,849


**Recommendation.** A single 45-minute promise is the right choice for most of
Rosa's business: in 8 of the 12 zone/time-block pairs, 45 minutes already
maximizes net profit. The exceptions are the busy evening segments, where
heavy demand makes deliveries slow and late orders expensive ($26.20 each).
Rosa should promise 55 minutes in Far West on Fri/Sat evenings, and 50 minutes
in North and Central on Fri/Sat evenings and in Far West on weekday evenings.
These are the same segments with the highest late rates in Part I(b). The most
important change is Far West Fri/Sat: at 45 minutes Rosa currently loses $127
there over four weeks, while a 55-minute promise earns $1,212. Overall,
segment-specific promises raise four-week net profit from $16,468 to $18,317,
a gain of about $1,849 (≈11%), without hiring additional drivers. Because we
tested promises in 5-minute steps, a finer search (e.g. 1-minute steps around
each recommendation) could fine-tune these numbers, but 5-minute promises are
also easier to communicate to customers.